In [1]:
import os
import pandas as pd
import re

In [2]:
import os
path = os.path.dirname(os.path.dirname(os.getcwd()))
print(path)

c:\Universidad\1\Universidad\TFG\TFG


In [3]:
metadata = pd.read_excel(path+'/transcripts/Discourse-UWO/metadata.xlsx',index_col=0)

In [4]:
idx_psico= metadata.index[metadata['PatientCat']==1].tolist()

In [5]:
print(f"Transcriptions not processed: {idx_psico[:5]}")

Transcriptions not processed: [1, 2, 3, 4, 5]


## Extraemos toda las las entrevistas y las guardamos con el siguiente formato:
```json
{"filename1":[q1,q2,q3,q4,q5,q6,q7],"filename2": ...}
```
Siendo qx una lista con todas las intervenciones de esa pregunta/seccion

In [6]:
def extract_interviews(folder_path,followup=False):
    todas_interviews = {}
    pat = {}
    for filename in os.listdir(folder_path):
        if int(filename.split('.')[0]) not in idx_psico[5:]:
            continue
        if filename.endswith('.cha') and filename not in ["007.cha"]:
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                if followup:
                    filename = filename + "_followup"
                groups = []
                group = []
                content = file.readlines()#[10:]
                content_no_mod = content.copy()
                line_index = [i for i, line in enumerate(content) if line.startswith('@G:	Free')]
                if line_index is None or len(line_index)==0:
                    continue
                else:
                    line_index = line_index[0]
                content = content[line_index+1:]
                for row in content:
                    if row.startswith('@'):
                        if len(group)==0:
                            pass
                        else:
                            group = [re.sub(r'\t',' ',line.strip()) for line in group if line.strip() and not line.startswith('@')]
                            group = [re.sub(r'.*$', '', line) for line in group]
                            group = [line[1:].strip() for line in group if line.startswith('*INV:') or line.startswith('*PAR:')]
                            groups.append(group)
                            group = []
                            if pat.get(filename) is None:
                                pat[filename] = 1
                            else:
                                pat[filename] +=1
                    elif row.startswith('%'):
                        pass
                    else:
                        if pat.get(filename) is None:
                            pat[filename] = 1
                        else:
                            pat[filename] +=1
                        group.append(row.strip())        
                todas_interviews[filename] = groups
    todas_interviews = {k:v for k,v in todas_interviews.items() if len(v)==7}
    return todas_interviews,pat


In [7]:
folder_path = path +'/transcripts/Discourse-UWO/Baseline'
todas_interviews, pat_baseline = extract_interviews(folder_path)
folder_path_followup = path +'/transcripts/Discourse-UWO/FollowUp'
todas_interviews_followup, pat_followup = extract_interviews(folder_path_followup, followup=True)

In [8]:
#Juntamos todos los datos en un solo diccionario
it_base = list(todas_interviews.items())
it_followup = list(todas_interviews_followup.items())
it_base.extend(it_followup)
todas_interviews = dict(it_base)


In [15]:
#Convertimos el formato a un diccionario con la siguiente estructura: 
# {filename: [{"part":i, "utterances":[(speaker, utterance),...]}]}
todas_processed = {}
for file in todas_interviews:
    todas_processed[file] = []
    for i,part in enumerate(todas_interviews[file]):
        aux = [(line[:3],line[4:].strip()) for i, line in enumerate(part)]
        todas_processed[file].append({"part":i, "utterances":aux})
print(f"Ejemplo: {todas_processed[list(todas_processed.keys())[0]][:5]}")

Ejemplo: [{'part': 0, 'utterances': [('INV', 'first .'), ('INV', 'can you tell me a bit about yourself .'), ('PAR', "&-um so I'm a woman ."), ('PAR', "I'm thirty nine years old ."), ('PAR', "I'm originally from Brazil ."), ('PAR', "I'm married and &-uh I live here ."), ('PAR', 'I moved here to Canada seven years ago .'), ('PAR', 'and &-uh I work I studied &-um pretty much &-like hotel management .'), ('PAR', "my background's &-like in the tourism area ."), ('PAR', '&-um but now I work with &-uh in the home decor industry .'), ('INV', 'okay .'), ('INV', 'perfect .'), ('INV', '&-um so do you wanna tell us a bit more about &-like your work and some of the things you do .'), ('INV', 'day to day .'), ('PAR', "&-uh it's basically &-uh on sales ."), ('PAR', '&-um so every day we have a meeting in the morning .'), ('PAR', 'we talk about our numbers and what are the strategies we can use &-uh on a daily basis .'), ('PAR', '&-um &+we I work with three other &+salesperson people and &-um pretty m


### Agrupando todas las intervenciones del mismo hablante
Con la siguiente estructura:
```json
{"filename":[{"part":0,"utterances":[{"position":0,"speaker":xxx,"text":xxx},{"position":1,"speaker":xxx,"text":xxx}...]}...]}
```
Por cada archivo hay varias partes y por cada parte hay varias intervenciones

In [16]:
todas_agrupadas = {}
for file, partes in todas_processed.items():
    todas_agrupadas[file] = []
    for parte in partes:
        num_parte,texto = parte["part"], parte["utterances"]
        agrupadas = []
        if not texto:
            print(file)
            todas_agrupadas[file] = agrupadas
            continue
        num =0
        actual_speaker, actual_text = texto[0]
        for i, (speaker, text) in enumerate(texto[1:]):
            if speaker == actual_speaker:
                actual_text += "\n " + text
            else:
                agrupadas.append({"position": num, "speaker": actual_speaker, "text": actual_text})
                actual_speaker, actual_text = speaker, text
                num += 1
        # Añade la última intervención
        agrupadas.append({"position": num, "speaker": actual_speaker, "text": actual_text})  
        todas_agrupadas[file].append({"part": num_parte, "utterances": agrupadas})

In [17]:
#Convertimos el diccionario a un dataframe con las siguientes 
# columnas: filename, part, position, role, answer
prep_df = []
for file, intervenciones in todas_agrupadas.items():
    for parte in intervenciones:
        prep_df.extend([(file,parte["part"],i["position"],i["speaker"],i["text"]) for i in parte["utterances"]])
df = pd.DataFrame(prep_df,columns=['filename','part','position','role','answer'])
df

,filename,part,position,role,answer
0,121.cha,0,0,INV,first .\n can you tell me a bit about yourself .
1,121.cha,0,1,PAR,&-um so I'm a woman .\n I'm thirty nine years ...
2,121.cha,0,2,INV,okay .\n perfect .\n &-um so do you wanna tell...
3,121.cha,0,3,PAR,&-uh it's basically &-uh on sales .\n &-um so ...
4,121.cha,0,4,INV,perfect .\n okay .\n &-um and do you wanna tel...
...,...,...,...,...,...
7115,028.cha_followup,6,2,INV,alright so I'll get you just to review the sto...
7116,028.cha_followup,6,3,PAR,okay .
7117,028.cha_followup,6,4,INV,alright go ahead and tell me the story .
7118,028.cha_followup,6,5,PAR,&-um so it was a very hot day .\n &-uh there w...


In [ ]:
df_par = df[df['role'] == 'PAR'].copy()
df_par['question'] = df_par.apply(
    lambda row: df.loc[(df['filename'] == row['filename']) & (df['role'] == 'INV') & (df.index < row.name), 'answer'].iloc[-1]
    if not df.loc[(df['filename'] == row['filename']) & (df['role'] == 'INV') & (df.index < row.name), 'answer'].empty else None,
    axis=1
)
df_par.reset_index(drop=True, inplace=True)


In [19]:
df_par

,filename,part,position,role,answer,question
0,121.cha,0,1,PAR,&-um so I'm a woman .\n I'm thirty nine years ...,first .\n can you tell me a bit about yourself .
1,121.cha,0,3,PAR,&-uh it's basically &-uh on sales .\n &-um so ...,okay .\n perfect .\n &-um so do you wanna tell...
2,121.cha,0,5,PAR,&-uh it's a family of five .\n &-um my father ...,perfect .\n okay .\n &-um and do you wanna tel...
3,121.cha,0,7,PAR,mhm .,yeah .\n yeah .\n perfect .\n okay .\n &-um so...
4,121.cha,0,9,PAR,&-um I [/] I used to &-like more going abroad ...,&-um so I'm not sure how much traveling you've...
...,...,...,...,...,...,...
3316,028.cha_followup,5,3,PAR,&-um yeah okay &-um so I do dream about patien...,yeah do you wanna just tell me one of those dr...
3317,028.cha_followup,5,5,PAR,yeah .\n yeah exactly .,only time will tell .\n yeah .
3318,028.cha_followup,6,1,PAR,but it was a hot day .\n a thirsty bird was lo...,okay last section here .\n so I have a one pag...
3319,028.cha_followup,6,3,PAR,okay .,alright so I'll get you just to review the sto...


In [ ]:
df_par['patient'] = df_par['filename'].apply(lambda x: int(x.split('.')[0]))
df_par['schizophrenia'] = 0
df_par.sample(5)

,filename,part,position,role,answer,question,patient,schizophrenia
407,070.cha,6,1,PAR,okay .\n it was a hot day .\n a thirsty bird w...,&-um so last thing so I have a one page story ...,70,0
1146,095.cha,1,7,PAR,mhm .,mhm .\n cool all right .\n and so in addition ...,95,0
645,123.cha,1,3,PAR,&-um okay .\n I think I can .,mhm .,123,0
399,070.cha,4,5,PAR,so it's this old lighthouse keeper on an islan...,go ahead .\n yeah .,70,0
1389,069.cha,2,7,PAR,currently I still like sometimes like have man...,so currently &-um like .,69,0


In [21]:
import json
data = df_par.to_dict(orient='records')
with open(path +'/data/ordered_healthy_discourse_qa.json','w',encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print(f"Wrote {path}/data/ordered_healthy_discourse_qa.json with {len(data)} records")

Wrote /home/pablo/Desktop/TFG/data/ordered_healthy_discourse_qa.json with 3321 records


In [ ]:
from datasets import load_dataset, Dataset
import numpy as np

print("Loading SFT dataset...")

dataset = load_dataset('json', data_files=path+'/data/ordered_healthy_discourse_qa.json')

# Añadir columna de paciente para agrupar
df_temp = df_par.copy()

# Obtener lista única de pacientes y hacer shuffle
unique_patients = df_temp['patient'].unique()
np.random.seed(42)
np.random.shuffle(unique_patients)

# Dividir pacientes en train (80%), test (10%), eval (10%)
n_patients = len(unique_patients)
n_train = int(n_patients * 0.8)
n_test = int(n_patients * 0.1)

train_patients = unique_patients[:n_train]
test_patients = unique_patients[n_train:n_train+n_test]
eval_patients = unique_patients[n_train+n_test:]

# Crear datasets basados en pacientes
train_df = df_temp[df_temp['patient'].isin(train_patients)].to_dict('list')
test_df = df_temp[df_temp['patient'].isin(test_patients)].to_dict('list')
eval_df = df_temp[df_temp['patient'].isin(eval_patients)].to_dict('list')

train_dataset_sft = Dataset.from_dict(train_df)
test_dataset_sft = Dataset.from_dict(test_df)
eval_dataset_sft = Dataset.from_dict(eval_df)

train_dataset_sft.to_json(path+'/data/ordered_healthy_train_dataset.json')
test_dataset_sft.to_json(path+'/data/ordered_healthy_test_dataset.json')
eval_dataset_sft.to_json(path+'/data/ordered_healthy_eval_dataset.json')

print("SFT Dataset loaded (grouped by patient):")
print(f" Train samples: {len(train_dataset_sft)} ({len(train_patients)} patients)")
print(f" Test samples: {len(test_dataset_sft)} ({len(test_patients)} patients)")
print(f" Eval samples: {len(eval_dataset_sft)} ({len(eval_patients)} patients)")
print(f"\n Single Sample: {train_dataset_sft[0]}")

Loading SFT dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

SFT Dataset loaded (grouped by patient):
 Train samples: 2756 (38 patients)
 Test samples: 239 (4 patients)
 Eval samples: 326 (6 patients)

 Single Sample: {'filename': '121.cha', 'part': 0, 'position': 1, 'role': 'PAR', 'answer': "&-um so I'm a woman .\n I'm thirty nine years old .\n I'm originally from Brazil .\n I'm married and &-uh I live here .\n I moved here to Canada seven years ago .\n and &-uh I work I studied &-um pretty much &-like hotel management .\n my background's &-like in the tourism area .\n &-um but now I work with &-uh in the home decor industry .", 'question': 'first .\n can you tell me a bit about yourself .', 'patient': 121, 'schizophrenia': 0}
